In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr
from src.utils import create_df, merge_dfs, filter_events_by_time_window, build_numeric_feature_table, build_event_occurrence_feature_table
from config.config import MIMIC_DIR, DATASET_DIR
from config.map_auto import CHARTEVENTS_FEATURES_MAP, DATETIMEEVENTS_FEATURES_MAP, LABEVENTS_FEATURES_MAP, OUTPUTEVENTS_FEATURES_MAP, INPUTEVENTS_MV_FEATURES_MAP
import time

spark = SparkSession.builder \
    .appName("MIMIC_III_LengthOfStay_Project") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.debug.maxToStringFields", 100) \
    .getOrCreate()

print(f"Spark Session Initialized! Version: {spark.version}")
start = time.time()

Spark Session Initialized! Version: 4.1.1
Spark Session Initialized! Version: 4.1.1


## Patient DF

In [2]:
patients_df = create_df(f"{MIMIC_DIR}/PATIENTS.csv", columns=["SUBJECT_ID", "GENDER", "DOB", "EXPIRE_FLAG"])

## Merging with Admissions

In [3]:
admissions_df = create_df(f"{MIMIC_DIR}/ADMISSIONS.csv", columns=["SUBJECT_ID", "HADM_ID", "ADMISSION_TYPE", "DIAGNOSIS", "ADMITTIME"], describe=False)

merged_df = merge_dfs(patients_df, admissions_df, column="SUBJECT_ID")

## Merging with ICUStay

In [4]:
icustays_df = create_df(f"{MIMIC_DIR}/ICUSTAYS.csv", columns=["HADM_ID", "ICUSTAY_ID", "INTIME", "LOS"])
# icustays_df = icustays_df.select("HADM_ID", "ICUSTAY_ID", "FIRST_CAREUNIT",  "LAST_CAREUNIT", "INTIME", "LOS")

merged_df = merge_dfs(merged_df, icustays_df, column="HADM_ID")

As we are goint to analyse the first 24h hours i created a new column, with this date just to simplify the featuring later

In [5]:
merged_df = merged_df.withColumn("END_FIRST_24H", expr("INTIME + INTERVAL 24 HOURS"))

time_reference_df = merged_df.select(
    "SUBJECT_ID",
    "ICUSTAY_ID",
    "ADMITTIME",
    "INTIME",
    "END_FIRST_24H"
).dropDuplicates(["ICUSTAY_ID"])

## Merging with Chartevents

In [6]:
chartevents_df = create_df(f"{MIMIC_DIR}/CHARTEVENTS.csv", columns=["ICUSTAY_ID", "ITEMID", "CHARTTIME", "VALUENUM"], describe=False)

Aparently the variables WARNING and ERROR provides information about data quality, maybe we should only use data without errors, but i think this is optional

### The first 24 hours

In [7]:
chartevents_24h = filter_events_by_time_window(
    events_df=chartevents_df,
    reference_df=time_reference_df
)

1. Define which clinical signals we want (e.g., heart rate, BP, etc.)
2. Map each ITEMID to the clinical variable name
3. Filter only the relevant events within the first 24 hours
4. Convert ITEMID to feature_name (e.g., 211 to “heart_rate”)
5. Group by patient and variable → calculate min, avg, max
6. Transform from “long” to “wide” format (pivot)
7. Generate a final table with:
   - one row per patient (ICUSTAY_ID)
   - multiple columns (features)

In [8]:
chartevents_features_df = build_numeric_feature_table(
    events_df=chartevents_24h,
    features_map=CHARTEVENTS_FEATURES_MAP,
    id_column="ICUSTAY_ID",
    code_column="ITEMID",
    value_column="VALUENUM",
    metrics=("min", "avg", "max")
)

In [9]:
chartevents_features_df.show()

+----------+----------------+------------------+-----------+-------------+-------------+--------------+-------------------+--------------+----------+-------+-----------------+--------------+--------+--------------------+--------+---------------+------------------+----------------+-----------------+------------------+-------------------+------------------+------------------+------------------+--------------+-------------------+------------------+----------+-----------------+------------------+------------------+--------+--------------------+-----------------+------------------+------------------+------------------+-----------------+----------------+------------------+-----------+-------------+-------------+--------------+-------------------+--------------+----------+-------+------------------+--------------+--------+--------------------+--------+---------------+------------------+----------------+-----------------+
|ICUSTAY_ID|diastolic_bp_min|          fio2_min|gcs_eye_min|gcs_motor_mi

In [10]:
merged_df = merge_dfs(merged_df, chartevents_features_df, column="ICUSTAY_ID")

## Merging datetime events

In [11]:
datetimeevents_df = create_df(f"{MIMIC_DIR}/DATETIMEEVENTS.csv", describe=False)

! Talvez em vez de pegar primeiras 24h sjam mlr pegar desde que entrou no hospital (admissions)

In [12]:
datetimeevents_24h = filter_events_by_time_window(
    events_df=datetimeevents_df,
    reference_df=time_reference_df,
    join_key="ICUSTAY_ID",
    event_time_col="CHARTTIME",
    start_col="INTIME",
    end_col="END_FIRST_24H",
    value_col="VALUE",
    drop_null_values=True
)

In [13]:
datetimeevents_features_df = build_event_occurrence_feature_table(
    events_df=datetimeevents_24h,
    features_map=DATETIMEEVENTS_FEATURES_MAP,
    id_column="ICUSTAY_ID",
    code_column="ITEMID"
)

In [14]:
merged_df = merge_dfs(merged_df, datetimeevents_features_df, column="ICUSTAY_ID")

## Merging Lab Events

In [15]:
labevents_df = create_df(f"{MIMIC_DIR}/LABEVENTS.csv", describe=False)

In [16]:
labevents_window = filter_events_by_time_window(
    events_df=labevents_df,
    reference_df=time_reference_df,
    join_key="SUBJECT_ID",
    event_time_col="CHARTTIME",
    start_col="ADMITTIME",
    end_col="END_FIRST_24H",
    value_col="VALUENUM",
    drop_null_values=True
)

In [17]:
labevents_features_df = build_numeric_feature_table(
    events_df=labevents_window,
    features_map=LABEVENTS_FEATURES_MAP,
    id_column="ICUSTAY_ID",
    code_column="ITEMID",
    value_column="VALUENUM",
    time_column="CHARTTIME",
    metrics=("latest",)
)

In [18]:
merged_df = merge_dfs(merged_df, labevents_features_df, column="ICUSTAY_ID")

## Merging Output Events

In [19]:
outputevents_df = create_df(f"{MIMIC_DIR}/OUTPUTEVENTS.csv", describe=False)

In [20]:
outputevents_window = filter_events_by_time_window(
    events_df=outputevents_df,
    reference_df=time_reference_df,
    join_key="ICUSTAY_ID",
    event_time_col="CHARTTIME",
    start_col="INTIME",
    end_col="END_FIRST_24H",
    value_col="VALUE",
    drop_null_values=True
)

In [21]:
outputevents_features_df = build_numeric_feature_table(
    events_df=outputevents_window,
    features_map=OUTPUTEVENTS_FEATURES_MAP,
    id_column="ICUSTAY_ID",
    code_column="ITEMID",
    value_column="VALUE",
    time_column="CHARTTIME",
    metrics=("sum", "count")
)

In [22]:
merged_df = merge_dfs(merged_df, outputevents_features_df, column="ICUSTAY_ID")

## Merging Input Events

In [23]:
inputevents_mv_df = create_df(f"{MIMIC_DIR}/INPUTEVENTS_MV.csv", describe=False)

In [24]:
inputevents_mv_window = filter_events_by_time_window(
    events_df=inputevents_mv_df,
    reference_df=time_reference_df,
    join_key="ICUSTAY_ID",
    event_time_col="STARTTIME",
    start_col="INTIME",
    end_col="END_FIRST_24H",
    value_col="AMOUNT",
    drop_null_values=True
)

In [25]:
inputevents_mv_features_df = build_numeric_feature_table(
    events_df=inputevents_mv_window,
    features_map=INPUTEVENTS_MV_FEATURES_MAP,
    id_column="ICUSTAY_ID",
    code_column="ITEMID",
    value_column="AMOUNT",
    time_column="STARTTIME",
    metrics=("sum", "count")
)

In [26]:
merged_df = merge_dfs(merged_df, inputevents_mv_features_df, column="ICUSTAY_ID")

In [28]:
from pyspark.sql.types import TimestampType
from pyspark.sql.functions import col, date_format
import os

os.makedirs(DATASET_DIR, exist_ok=True)
timestamp_cols = [
    field.name
    for field in merged_df.schema.fields
    if isinstance(field.dataType, TimestampType)
]

safe_df = merged_df

for column_name in timestamp_cols:
    safe_df = safe_df.withColumn(
        column_name,
        date_format(col(column_name), "yyyy-MM-dd HH:mm:ss")
    )

final_df = safe_df.toPandas()
final_df.to_csv(f"{DATASET_DIR}/final.csv", index=False)
print(f"Run time: {time.time()-start}")

Run time: 1203.0530560016632
